In [112]:
!pip -q install python-docx

import re, json, hashlib
from pathlib import Path
from docx import Document


In [114]:
docx_path = "DA knowledge Articles.docx"  # in Colab you’ll upload / mount Drive
doc = Document(docx_path)

paras = [p.text.strip() for p in doc.paragraphs]
nonempty = [p for p in paras if p]

print("Total paragraphs:", len(paras))
print("Non-empty paragraphs:", len(nonempty))
print("\n--- Sample ---")
for i, t in enumerate(nonempty[:15]):
    print(f"{i+1:02d}. {t[:140]}")


Total paragraphs: 327
Non-empty paragraphs: 211

--- Sample ---
01. DA knowledge Articles
02. Departure Process:
03. Curb Side / Landside

Visitors can only drop off passengers in this area, but they are not allowed to park or stop for more than 10 minutes;
04. Feedback:

Passengers may be fined for long stops in this area; this area is controlled by Dubai Police, so customers are required to contac
05. Free trolleys are available at the entrance of the departure area
06. Porter service - no prior reservation required. This is a paid service.
07. Baggage wrapping and baggage weighing services. This is a paid service.
08. Feedback:

Passengers need to ask the service provider for fees before using the above services.
09. Check-in area:
10. If a passenger travels with a minor will need to contact the airline as the requirements for traveling with unaccompanied minors vary by air
11. Other airlines operating in T1 and T2 

Check-in Counter: The check-in staff (Dnata) is responsible for es

In [115]:
def is_heading(text: str) -> bool:
    # Heuristic: short-ish line ending with ":" and not too sentence-like
    if not text:
        return False
    if len(text) > 80:
        return False
    if not text.endswith(":"):
        return False
    # avoid time stamps / random punctuation
    if text.count(".") > 1:
        return False
    return True

PHONE_RE = re.compile(r"(\+?\d[\d\s\-\(\)]{7,}\d)")
URL_RE = re.compile(r"(https?://\S+)")
EMAIL_RE = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")

def contains_contact(text: str) -> bool:
    return bool(PHONE_RE.search(text) or EMAIL_RE.search(text))

def contains_url(text: str) -> bool:
    return bool(URL_RE.search(text))


In [116]:
def make_id(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]

blocks = []
heading_path = []
buffer = []

def flush():
    global buffer
    if not buffer:
        return
    text = "\n\n".join(buffer).strip()
    if not text:
        buffer = []
        return
    block = {
        "doc_id": "da_articles_v1",
        "block_id": make_id("|".join(heading_path) + "||" + text[:200]),
        "heading_path": [h.rstrip(":") for h in heading_path if h],
        "text": text,
        "contains_contact": contains_contact(text),
        "contains_url": contains_url(text),
    }
    blocks.append(block)
    buffer = []

for t in nonempty:
    # treat first title line as a doc title, not a heading
    if t.lower().strip() in ["da knowledge articles"]:
        continue

    if is_heading(t):
        flush()
        # simple rule: headings reset to a single-level path
        # (we’ll improve heading depth later if needed)
        heading_path = [t]
    else:
        buffer.append(t)

flush()

print("Blocks created:", len(blocks))
print("\nExample block:\n", json.dumps(blocks[0], indent=2)[:1200])


Blocks created: 18

Example block:
 {
  "doc_id": "da_articles_v1",
  "block_id": "09e6912560db",
  "heading_path": [
    "Departure Process"
  ],
  "text": "Curb Side / Landside\n\nVisitors can only drop off passengers in this area, but they are not allowed to park or stop for more than 10 minutes; if this time passes, this may result in a fine from the police.\nTherefore, visitors could use the car park to avail themselves of the shorter tariff bands for passenger pick-up. Prices are published on the website. The link is available in the FAQ document. We also have a pre-booking parking service where visitors can book a space if they need this service.\n\nFeedback:\n\nPassengers may be fined for long stops in this area; this area is controlled by Dubai Police, so customers are required to contact Dubai Police for further clarification on this.\n\nServices available before the check-in area\n\nFree trolleys are available at the entrance of the departure area\n\nPorter service - no prio

In [117]:
out_path = Path("processed_blocks.jsonl")
with out_path.open("w", encoding="utf-8") as f:
    for b in blocks:
        f.write(json.dumps(b, ensure_ascii=False) + "\n")

print("Saved:", out_path.resolve(), "lines:", len(blocks))


Saved: /content/processed_blocks.jsonl lines: 18


In [118]:
import json
from pathlib import Path

blocks = []
with open("processed_blocks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        blocks.append(json.loads(line))

len(blocks)


18

In [119]:
INTERNAL_PATTERNS = [
    r"\bFYI\b",
    r"no need to share",
    r"do not share",
    r"highlighted",
    r"internal",
]

def is_internal_only(text: str) -> bool:
    t = text.lower()
    return any(re.search(p, t) for p in INTERNAL_PATTERNS)


In [127]:
def is_subsection_header(line: str) -> bool:
    # explicit known patterns
    if line.lower().startswith("feedback"):
        return True
    if line.lower().startswith("services available"):
        return True

    # short title-like lines
    if len(line) < 80 and line.endswith(":"):
        return True

    # section-style with slashes
    if "/" in line and len(line) < 60:
        return True

    return False


In [128]:
chunks = []
chunk_counter = 0

def process_block(block):
    global chunk_counter
    lines = [l.strip() for l in block["text"].split("\n") if l.strip()]
    current_subsection = None
    buffer = []

    def flush_chunk():
        nonlocal buffer, current_subsection
        global chunk_counter
        if not buffer:
            return
        text = " ".join(buffer).strip()
        if not text:
            buffer = []
            return
        chunk = {
            "chunk_id": f"c_{chunk_counter:06d}",
            "heading_path": block["heading_path"],
            "subsection": current_subsection,
            "text": text,
            "internal_only": is_internal_only(text),
            "contains_contact": contains_contact(text),
            "contains_url": contains_url(text),
        }
        chunks.append(chunk)
        chunk_counter += 1
        buffer = []

    for line in lines:
        if is_subsection_header(line):
            flush_chunk()
            current_subsection = line.rstrip(":")
        else:
            buffer.append(line)

    flush_chunk()

for block in blocks:
    process_block(block)

len(chunks)

58

In [129]:
import textwrap

# How many internal-only chunks?
internal_chunks = [c for c in chunks if c["internal_only"]]
print("Total chunks:", len(chunks))
print("Internal-only chunks:", len(internal_chunks))

# Show a few sample chunks
for c in chunks[:5]:
    print("\n---")
    print("Heading:", " > ".join(c["heading_path"]))
    print("Subsection:", c["subsection"])
    print("Internal:", c["internal_only"])
    print(textwrap.fill(c["text"][:400], 90))

Total chunks: 58
Internal-only chunks: 1

---
Heading: Departure Process
Subsection: Curb Side / Landside
Internal: False
Visitors can only drop off passengers in this area, but they are not allowed to park or
stop for more than 10 minutes; if this time passes, this may result in a fine from the
police. Therefore, visitors could use the car park to avail themselves of the shorter
tariff bands for passenger pick-up. Prices are published on the website. The link is
available in the FAQ document. We also have a pre-book

---
Heading: Departure Process
Subsection: Feedback
Internal: False
Passengers may be fined for long stops in this area; this area is controlled by Dubai
Police, so customers are required to contact Dubai Police for further clarification on
this.

---
Heading: Departure Process
Subsection: Services available before the check-in area
Internal: False
Free trolleys are available at the entrance of the departure area Porter service - no
prior reservation required. This is a p

In [130]:
out_path = Path("processed_chunks.jsonl")
with out_path.open("w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved:", out_path.resolve())


Saved: /content/processed_chunks.jsonl


In [131]:
!pip -q install sentence-transformers faiss-cpu


In [132]:
import json

chunks = []
with open("processed_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        if not c["internal_only"]:
            chunks.append(c)

len(chunks)


57

In [147]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings.shape

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(57, 384)

In [134]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity (with normalized vectors)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 57


In [135]:
import pickle
from pathlib import Path

Path("vector_store").mkdir(exist_ok=True)

faiss.write_index(index, "vector_store/da_faiss.index")

with open("vector_store/chunks_metadata.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Vector store saved")


Vector store saved


In [148]:
def retrieve(query, top_k=5):
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[idx]
        results.append({
            "score": float(score),
            "heading": " > ".join(c["heading_path"]),
            "subsection": c["subsection"],
            "text": c["text"][:300]
        })
    return results

# Try real airport questions
for r in retrieve("How long can I stop at the drop off area?", 3):
    print("\n---")
    print("Score:", r["score"])
    print("Section:", r["heading"], "|", r["subsection"])
    print(r["text"])


---
Score: 0.4754442870616913
Section: Departure Process | Curb Side / Landside
Visitors can only drop off passengers in this area, but they are not allowed to park or stop for more than 10 minutes; if this time passes, this may result in a fine from the police. Therefore, visitors could use the car park to avail themselves of the shorter tariff bands for passenger pick-up. Pri

---
Score: 0.3020893931388855
Section: Departure Process | Feedback
Passengers may be fined for long stops in this area; this area is controlled by Dubai Police, so customers are required to contact Dubai Police for further clarification on this.

---
Score: 0.27011242508888245
Section: Feedback | None
Check-in counter closed -As per the rules and regulations followed at the airport all check-in counters close 1 hour before departure time. It is the passenger’s responsibility to reach the airport at least 3 hours before the flight time to allow adequate time to complete the check-in process. Even 


In [149]:
print("Porter service free at Dubai airport?: ", retrieve("Is porter service free at Dubai airport?", 3))
print("How early can I check in with Emirates?: ", retrieve("How early can I check in with Emirates?", 3))
print("Who should I contact for immigration issues?: ", retrieve("Who should I contact for immigration issues?", 3))

Porter service free at Dubai airport?:  [{'score': 0.5796027183532715, 'heading': 'Departure Process', 'subsection': 'Services available before the check-in area', 'text': 'Free trolleys are available at the entrance of the departure area Porter service - no prior reservation required. This is a paid service. Baggage wrapping and baggage weighing services. This is a paid service.'}, {'score': 0.4889163672924042, 'heading': 'For more details, please find the below details', 'subsection': 'Meet and Greet Related (Free Lounges)', 'text': 'Depending on the airline, class of service, and airport, passengers may only have access to one lounge, or they may have several to choose from. Therefore, passengers need to check eligibility to enter the lounge with the airlines or service providers either for free or for a fee. Passengers can vis'}, {'score': 0.47520601749420166, 'heading': 'Departure Process', 'subsection': 'Feedback', 'text': 'Passengers may be fined for long stops in this area; thi

In [179]:
SYSTEM_PROMPT = """
You are a Dubai Airports Customer Service Representative.

Rules:
- Answer ONLY using the provided airport knowledge.
- Do NOT guess or add information.
- If the answer is not found, say so clearly and guide the passenger to the correct authority.
- Be polite, professional, and concise.
- Do NOT mention internal notes or feedback.
- If clarification is required (terminal, airline, arrival/departure), ask ONE clear question.

Answer format:
1. Direct answer (1–3 sentences)
2. Next steps (bullet points if needed)
3. Contact or authority (only if mentioned in context)
"""


In [139]:
def build_context(retrieved_chunks, max_chars=2000):
    context = []
    total = 0

    for c in retrieved_chunks:
        block = f"""
SECTION: {c['heading']}
SUBSECTION: {c['subsection']}
CONTENT: {c['text']}
"""
        if total + len(block) > max_chars:
            break
        context.append(block)
        total += len(block)

    return "\n".join(context)


In [140]:
def rag_answer(query, retrieved):
    context = build_context(retrieved)

    prompt = f"""
{SYSTEM_PROMPT}

AIRPORT KNOWLEDGE:
{context}

PASSENGER QUESTION:
{query}

ANSWER:
"""
    return prompt


In [141]:
retrieved = retrieve("How long can I stop at the drop off area?", 3)
print(rag_answer("How long can I stop at the drop off area?", retrieved))




You are a Dubai Airports Customer Service Representative.

Rules:
- Answer ONLY using the provided airport knowledge.
- Do NOT guess or add information.
- If the answer is not found, say so clearly and guide the passenger to the correct authority.
- Be polite, professional, and concise.
- Do NOT mention internal notes or feedback.
- If clarification is required (terminal, airline, arrival/departure), ask ONE clear question.

Answer format:
1. Direct answer (1–3 sentences)
2. Next steps (bullet points if needed)
3. Contact or authority (only if mentioned in context)


AIRPORT KNOWLEDGE:

SECTION: Departure Process
SUBSECTION: Curb Side / Landside
CONTENT: Visitors can only drop off passengers in this area, but they are not allowed to park or stop for more than 10 minutes; if this time passes, this may result in a fine from the police. Therefore, visitors could use the car park to avail themselves of the shorter tariff bands for passenger pick-up. Pri


SECTION: Departure Process
SUBSE

In [164]:
!pip -q install transformers accelerate torch

In [178]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/phi-2"  # ✅ UNGATED

tokenizer = AutoTokenizer.from_pretrained(model_name)
local_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [180]:
def rag_answer_local(query, top_k=5):
    retrieved = retrieve(query, top_k)
    context = build_context(retrieved)

    prompt = f"""
{SYSTEM_PROMPT}

AIRPORT KNOWLEDGE:
{context}

PASSENGER QUESTION:
{query}

ANSWER:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(local_model.device)

    with torch.no_grad():
        outputs = local_model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)


    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Return only the answer part
    if "ANSWER:" in decoded:
        return decoded.split("ANSWER:")[-1].strip()
    return decoded.strip()


In [181]:
print(
    rag_answer_local("How long can I stop at the drop off area?")
)


1. You can only stop for 10 minutes at the drop off area.
2. If you exceed this time limit, you may be fined by the police.
3. Please use the car park to avail yourself of the shorter tariff bands for passenger pick-up.


In [182]:
print(rag_answer_local("When do check-in counters close?"))

1. Check-in counters close 1 hour before departure time.
2. Passengers should reach the airport at least 3 hours before the flight time to allow adequate time to complete the check-in process.
3. Contact the airline directly for correct information and necessary documents.


In [184]:
print(rag_answer_local("Who can use the smart gates at Dubai Airport?"))

1. Registered passengers with a height of 1.2 meters and above.
2. For more details, visit this link: https://www.gdrfad.gov.ae/en/smart-gate-inquiry
3. Passengers can also contact GDRFA(Immigration) https://www.gdrfad.gov.ae/en/contact-information


In [185]:
print(rag_answer_local("There is a missing entry stamp in my passport, what should I do?"))

1. Contact the General Directorate of Residency & Foreign Affairs/ GDRFA-D - Call Center (Amer Service) Toll-free number: 8005111 Outside U.A.E: +971 4 313 9999 Email: amer@gdrfad.gov.ae
2. Visit the Immigration Information Centre located in T3, exit 1 open 24/7
3. Provide the necessary details and follow the instructions given by the immigration officer.


In [187]:
print(rag_answer_local("What documents are required for a child traveling alone?"))

1. Passport
2. Valid visa (if required)
3. Proof of age (birth certificate or passport)
4. Contact information of parents/ guardians
5. Medical records (if required)

NEXT STEPS:
1. Contact the airline directly for specific requirements and documentation.
2. Visit the airline's website for detailed information.

CONTACT AUTHORITY:
The airline's customer service department.


In [188]:
print(rag_answer_local("My check-in counter was closed when I arrived, who is responsible?"))

1. The check-in counter was closed when you arrived.
2. The check-in counter closes 1 hour before departure time.
3. It is the passenger’s responsibility to reach the airport at least 3 hours before the flight time to allow adequate time to complete the check-in process.
4. Please contact the airline directly for further assistance.


In [189]:
!pip -q install gradio

In [190]:
import gradio as gr

def chat_with_rag(message, history):
    """
    message: latest user input
    history: list of (user, assistant) tuples
    """
    answer = rag_answer_local(message)
    history.append((message, answer))
    return history, history

In [191]:
with gr.Blocks(title="Dubai Airports Customer Service Assistant") as demo:

    gr.Markdown(
        """
        # ✈️ Dubai Airports – Customer Service Assistant
        Ask questions about check-in, immigration, smart gates, airport services, and more.

        **Note:** This assistant answers strictly based on official airport knowledge.
        """
    )

    chatbot = gr.Chatbot(
        height=400,
        label="Customer Service Chat"
    )

    user_input = gr.Textbox(
        placeholder="Ask your question here...",
        label="Passenger Question"
    )

    state = gr.State([])

    send_btn = gr.Button("Send")

    send_btn.click(
        fn=chat_with_rag,
        inputs=[user_input, state],
        outputs=[chatbot, state]
    )

    user_input.submit(
        fn=chat_with_rag,
        inputs=[user_input, state],
        outputs=[chatbot, state]
    )


/tmp/ipython-input-4115094621.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-4115094621.py:12: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [192]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b98001a79b8363124d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
